# 02. Multiagents: Subagents — Supervisor pattern

## Learning Objectives

- Design a three-tier architecture of Supervisor → subagent → tool
- Wrap subagent with `@tool` and expose it to supervisors as tool
- Understand HITL, ToolRuntime, and asynchronous/dispatch patterns.

## 2.1 Environment Setup

The Subagents pattern is a multi-agent architecture in which a central supervisor agent delegates tasks by calling specialized subagents like tool. On this laptop, we build a personal assistant system that handles calendars and email domains.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically enabled when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 2.2 Subagents Architecture Overview

The Subagents pattern consists of a **three-tier architecture**. The supervisor is responsible for all routing, subagent does not interact directly with the user, and returns results to the supervisor.

![Supervisor-subagent Hierarchy](../assets/images/supervisor_subagents.png)

| tier | Role | Features |
|------|------|------|
| **Low Level tool** | Direct call to external service (Calendar API, Email API) | Simple function wrapper |
| **subagent** | Domain-specific inference + tool combination | Expert system prompt, independent tool set |
| **Supervisor** | Decompose tasks, delegate, and aggregate results | Remember entire conversation, treat subagent = tool |

### Core Traits

- **Centralized Control**: All routing flows through the supervisor
- **Context Isolation**: subagent runs in a clean context window every time, preventing context bloat.
- **Parallel execution**: Multiple subagent can be called simultaneously in one turn
- **tool based calls**: Wrap subagent with `@tool` and expose it to the supervisor like a regular tool

### When to use

The subagent pattern is suitable when you manage multiple domains (calendar, email, CRM, etc.), but subagent does not need to interact directly with users, and you need centralized workflow management. For simple scenarios with few tool, a single agent is sufficient.

## 2.3 Low-level tool definitions

Defines the low-level tool, the bottom layer of the three-tier architecture. These tool are simple function wrappers that interact directly with external services (Calendar API, Email API). In actual production, it integrates with Google Calendar API, Email Service, etc., but here we use a stub implementation for learning.

Important points when designing tool:
- One tool is responsible for only one function (single responsibility principle)
- The same tool should not be assigned redundantly to multiple subagent
- Write the docstring clearly so that LLM can select tool correctly

In [ ]:
from langchain_core.tools import tool

@tool
def create_calendar_event(
    title: str, start_time: str, end_time: str,
    attendees: list[str] = None,
) -> str:
    """Create a new calendar event."""
    return f"이벤트 '{title}' 생성됨: {start_time} ~ {end_time}"

In [ ]:
@tool
def read_calendar_events(date: str) -> str:
    """Look up calendar events for a date (YYYY-MM-DD)."""
    return f"{date}에 이벤트가 없습니다."

In [ ]:
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email message."""
    return f"{to}에게 이메일 전송됨: '{subject}'"

@tool
def read_emails(folder: str = "inbox", limit: int = 10) -> str:
    """Read recent emails from a folder."""
    return f"{folder}에 이메일 3개"

In [ ]:
@tool
def search_emails(query: str, limit: int = 10) -> str:
    """Search for emails with your search term."""
    return f"'{query}' 검색 결과 2건"

## 2.4 Create subagent

Each subagent is created by `create_agent()` and has three core elements:

1. **Specialized system prompts**: Define domain-specific roles and behavioral guidelines
2. **Set tool by domain**: Separate concerns by assigning only tool for that domain
3. **`name` identifier**: Used by the supervisor to identify and call subagent

The recommended granularity of subagent is domain-level (calendar, email, etc.). Too much granularity increases the routing burden on the supervisor, while too much integration reduces the benefits of context isolation.

In [ ]:
from langchain.agents import create_agent

calendar_agent = create_agent(
    model="gpt-4.1",
    tools=[create_calendar_event, read_calendar_events],
    system_prompt="You are a calendar assistant. Use ISO 8601 date format.",
    name="calendar_agent",
)

In [ ]:
email_agent = create_agent(
    model="gpt-4.1",
    tools=[send_email, read_emails, search_emails],
    system_prompt="You are an email assistant. Write your message professionally.",
    name="email_agent",
)

## 2.5 Wrapping subagent with tool

The standard pattern for exposing subagent to a supervisor is to wrap it with a `@tool` decorator. Inside the wrapping function, call `subagent.invoke()` and return `content` of the last message.

Advantages of this pattern:
- From the supervisor’s perspective, subagent is treated the same as regular tool
- Changes to the internal implementation of subagent do not affect the supervisor.
- The input/output format can be freely customized in the wrapping function.

**Choose an input/output strategy**: You can pass just the query (simple) or the entire context (sophisticated). When returning results, you have the option of returning only the final result or returning the entire history.

In [ ]:
@tool("schedule_event", description="Calendar event scheduling")
def call_calendar(query: str) -> str:
    """Delegate calendar tasks to a calendar agent."""
    result = calendar_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=lf_config,
)
    return result["messages"][-1].content

In [ ]:
@tool("manage_email", description="Send, read and search email")
def call_email(query: str) -> str:
    """Delegate email tasks to email agents."""
    result = email_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=lf_config,
)
    return result["messages"][-1].content

## 2.6 Supervisor Agent Assembly

The supervisor is created by passing the wrapped subagent tool to `tools`. The supervisor's system prompts include task decomposition and delegation instructions.

Supervisor design considerations:
- **Error handling**: subagent failures must be handled gracefully by the supervisor.
- **Result Aggregation**: Consolidates results from multiple subagent to provide consistent responses to users
- **Approval Scope**: Applies HITL only to operations that change the state (sending an email, creating an event)

In [ ]:
supervisor = create_agent(
    model="gpt-4.1",
    tools=[call_calendar, call_email],
    system_prompt=(
        "You are a personal assistant. complex request"
        "Decompose it into subtasks and delegate them to appropriate agents."
    ),
)

## 2.7 Run test

```
User: "내일 2시 Sarah 미팅 잡고 초대 이메일 보내줘"
Supervisor → call_calendar → create_calendar_event
Supervisor → call_email → send_email
Supervisor: "미팅과 초대 이메일 완료"
```

In [ ]:
# response = supervisor.invoke({
#     "messages": [{"role": "user",
# "content": "I have a meeting with Sarah tomorrow at 2 o'clock."
# "Send me an invitation email."}]
# }, config=lf_config)
# print(response["messages"][-1].content)

## 2.8 HITL (Human-in-the-Loop) Integration

Combining `HumanInTheLoopMiddleware` and `checkpointer` allows you to request user approval before high-risk tool calling (sending emails, creating schedules, etc.). When the agent attempts to call a protected tool, execution is paused and reviewed by the user.

### Authorization response type

| Reply | Description | code |
|------|------|------|
| **Approve** | Run tool calling as is | `Command(resume="approve")` |
| **Edit** | Execute after modifying tool argument | `Command(resume={"type": "edit", "args": {...}})` |
| **Reject** | Cancel tool calling | `Command(resume={"type": "reject", "reason": "..."})` |

It is recommended to apply HITL only to operations that change state (send_email, create_calendar_event, etc.). It is not applied to read-only operations to reduce unnecessary friction.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

hitl = HumanInTheLoopMiddleware(interrupt_on={
    "schedule_event": {"allowed_decisions": ["approve", "edit", "reject"]},
    "manage_email": {"allowed_decisions": ["approve", "reject"]},
})

In [ ]:
supervisor_hitl = create_agent(
    model="gpt-4.1",
    tools=[call_calendar, call_email],
    checkpointer=InMemorySaver(),
    middleware=[hitl],
    system_prompt="You are a personal assistant.",
)

In [ ]:
from langgraph.types import Command

# result = supervisor_hitl.invoke(Command(resume="approve"), config=lf_config)
# result = supervisor_hitl.invoke(Command(resume={"type": "reject", "reason": "I changed my mind"}), config=lf_config)

## 2.9 Passing context (ToolRuntime)

`ToolRuntime` is a mechanism for passing runtime context (user ID, name, time zone, etc.) to tool without including it in the message. Instead of putting repetitive text in every prompt, you set the shared context with `ToolRuntime` just once.

The tool function accesses the context with the `runtime_context` keyword argument. Through this:
- User identity information may be used by tool (e.g. to automatically set sender email)
- Environment Setup (time zone, etc.) can be applied consistently
- Reduce token cost by reducing prompt length

In [ ]:
# ToolRuntime is the runtime context passing mechanism for LangChain v1 agents.
# This is a fallback in case it hasn't been released yet.
try:
    from langchain.runtime import ToolRuntime
except ImportError:
    # Simple fallback implementation if ToolRuntime is not yet released
    class ToolRuntime:
        """Fallback ToolRuntime for pre-release versions."""
        def __init__(self, context: dict):
            self.context = context
    print("ToolRuntime unreleased — use fallback stub")

runtime = ToolRuntime(context={
    "user_email": "me@example.com",
    "user_name": "Alice",
    "timezone": "Asia/Seoul",
})

In [ ]:
supervisor_ctx = create_agent(
    model="gpt-4.1",
    tools=[call_calendar, call_email],
    system_prompt="You are a personal assistant.",
)

In [ ]:
# Accessing runtime context in tool
@tool
def send_email_ctx(
    to: str, subject: str, body: str, *, runtime_context: dict
) -> str:
    """Send an email to the current user."""
    sender = runtime_context["user_email"]
    return f"{sender}에서 {to}로 이메일 전송됨: '{subject}'"

## 2.10 Asynchronous execution pattern

subagent has two execution modes:

| mode | Action | When to use |
|------|------|----------|
| **Synchronous** | Supervisor waits for subagent completion and then proceeds | When results are needed for next operation (default) |
| **Asynchronous** | Return Job ID immediately, run in background, view results later | Independent work, long-time work |

Asynchronous patterns follow the structure **Job ID → Status → Result**. If subagent returns the Job ID immediately, the supervisor can continue working on other tasks and retrieve the results later.

In [ ]:
import uuid
job_store = {}

@tool("schedule_async", description="Event scheduling (asynchronous)")
def call_calendar_async(query: str) -> str:
    """Starts an asynchronous calendar task and returns the task ID."""
    job_id = str(uuid.uuid4())[:8]
    job_store[job_id] = {"status": "done", "result": "Event created"}
    return f"작업 시작됨: {job_id}"

In [ ]:
@tool("check_job", description="Check status of asynchronous task")
def check_job(job_id: str) -> str:
    """Check the status of an asynchronous operation."""
    job = job_store.get(job_id, {"status": "not_found"})
    return f"상태: {job['status']}, 결과: {job.get('result')}"

## 2.11 Single Dispatch tool Pattern

There are two approaches to the tool pattern:

| pattern | Description | Advantages |
|------|------|------|
| **By agent tool** | Create a separate wrapping tool for each subagent | Detailed control, easy description customization |
| **Single Dispatch tool** | Call all subagent with one parameterized tool | Excellent scalability, independent addition/removal of subagent |

The single dispatch pattern specifies the target of the call with the `agent_name` parameter. Because subagent registered in the agent registry is called by looking up its name, subagent can be added or removed independently in distributed teams, providing excellent scalability.

In [ ]:
from typing import Literal

AGENT_REGISTRY = {"calendar": calendar_agent, "email": email_agent}

@tool("delegate", description="Delegate to a professional agent")
def dispatch(agent_name: Literal["calendar", "email"], query: str) -> str:
    """Routes the task to the specified subagent."""
    agent = AGENT_REGISTRY[agent_name]
    result = agent.invoke({"messages": [{"role": "user", "content": query}]}, config=lf_config)
    return result["messages"][-1].content

In [ ]:
supervisor_dispatch = create_agent(
    model="gpt-4.1",
    tools=[dispatch],
    system_prompt=(
        "Use delegate tool to route work."
        "Agent: 'calendar', 'email'."
    ),
)

## Summary

| Item | core |
|------|------|
| **Tier 3** | tool → subagent(`create_agent`) → Supervisor |
| **Wrapping** | `@tool` + `subagent.invoke()` → Return last content |
| **Quarantine** | subagent = clean context, only supervisor remembers full |
| **HITL** | `HumanInTheLoopMiddleware` + `InMemorySaver` |
| **Context** | `ToolRuntime(context={...})` → `runtime_context` |
| **Asynchronous** | Job ID → Status → Result pattern |
| **Dispatch** | Single `dispatch(agent_name, query)` tool |

### Next Steps
→ **[03_multi_agent_handoffs_router.ipynb](./03_multi_agent_handoffs_router.ipynb)**: Learn Handoffs and Router patterns.